# GenInherit-LLM: Phase 1 Data Preparation & Preprocessing
### Google Colab Automated Pipeline for Genomic & Pedigree Sharding

This notebook runs the complete **Phase 1** data preparation pipeline:
1. Mounts your Google Drive (`/MyDrive/geninherit/`)
2. Downloads public genomic datasets (ClinVar, HPO, MANE, ClinGen)
3. Generates 15,000 synthetic multigenerational family pedigrees (70+ archetypes)
4. Builds the 4,096-token **BioDSL** hybrid vocabulary and runs a round-trip sanity check
5. Preprocesses, serializes (PSTS-v1), attaches the Fourier numeric channel, and saves binary shards (`.pt`) directly to Google Drive
6. Audits tensor shapes and verifies readiness for Phase 2 base model pre-training

### Step 1: Mount Google Drive & Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
WORKSPACE_DRIVE = '/content/drive/MyDrive/geninherit'
LOCAL_RAW = '/content/raw_data'
SHARDS_DIR = f'{WORKSPACE_DRIVE}/data/shards'

os.makedirs(LOCAL_RAW, exist_ok=True)
os.makedirs(SHARDS_DIR, exist_ok=True)
os.makedirs(f'{WORKSPACE_DRIVE}/code', exist_ok=True)

print('✓ Google Drive connected successfully!')

### Step 2: Install Minimal Dependencies

In [ ]:
!pip install -q torch transformers tokenizers safetensors
print('✓ Dependencies installed.')

### Step 3: Download Public Genomic Datasets (~1.3 GB)
*(Downloads ClinVar variant classifications, HPO phenotypes, MANE canonical transcripts, and ClinGen validity)*

In [ ]:
!python /content/drive/MyDrive/geninherit/code/01_download_raw_data.py /content/raw_data

### Step 4: Generate 15,000 Multigenerational Family Pedigrees
*(Generates mathematically grounded families across AD, AR, XLR, and De Novo archetypes with longitudinal onset ages)*

In [ ]:
!python /content/drive/MyDrive/geninherit/code/02_synthetic_pedigree_generator.py \
    /content/raw_data/synthetic_pedigrees.json 15000

### Step 5: Build BioDSL 4,096-Token Vocabulary & Sanity Check
*(Creates `bioDSL_vocab.json` directly in Google Drive and verifies 100% round-trip tokenization)*

In [ ]:
!python /content/drive/MyDrive/geninherit/code/03_build_biodsl_tokenizer.py \
    /content/drive/MyDrive/geninherit/bioDSL_vocab.json

### Step 6: Preprocess & Pack into Binary Shards on Google Drive
*(Serializes to PSTS-v1 tokens, runs Wright-Malecot kinship recurrence, computes Fourier numeric channels, and saves `shard_XXXX.pt`)*

In [ ]:
!python /content/drive/MyDrive/geninherit/code/04_preprocess_and_shard.py \
    /content/raw_data/synthetic_pedigrees.json \
    /content/drive/MyDrive/geninherit/bioDSL_vocab.json \
    /content/drive/MyDrive/geninherit/data/shards \
    2500

### Step 7: Audit & Verify All Generated Shards
*(Checks tensor dtypes, active supervised tokens, and verifies readiness for Phase 2 training)*

In [ ]:
!python /content/drive/MyDrive/geninherit/code/05_verify_phase1_shards.py \
    /content/drive/MyDrive/geninherit/data/shards